# AI Resume Analyzer

## Objective

This notebook combines all previously developed modules into a single AI Resume Analysis pipeline.

The system performs the following tasks:

- Resume Category Prediction
- Semantic Resume Matching
- ATS Score Calculation
- Skill Gap Analysis
- Retrieval-Based Resume Analysis
- Personalized Feedback Generation
- Gemini AI Feedback (Optional)

This notebook represents the complete AI Resume Screening System.

# Step 1: Import Required Libraries

Import all required Python libraries.

In [3]:
import pandas as pd

import numpy as np

import pickle

import joblib

from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

import faiss

## Step 2: Load Saved Models

Load all saved machine learning models created in previous notebooks.

In [4]:
classifier = joblib.load("../models/resume_classifier.pkl")

tfidf = joblib.load("../models/tfidf_vectorizer.pkl")

label_encoder = joblib.load("../models/label_encoder.pkl")

print("All ML models loaded successfully.")

All ML models loaded successfully.


### Step 3: Load Resume Dataset

Load the processed resume dataset for analysis.

In [5]:
df = pd.read_csv("../data/final_feature_dataset.csv")

print(df.shape)

df.head()

(3500, 16)


,ResumeID,Category,Skills,Education,Experience,Clean_Text,Text,Source,Resume_Length,Word_Count,Sentence_Count,Skill_Count,Education_Length,Experience_Length,Source_Encoded,Category_Encoded
0,REAL_0001,Java Developer,"Python, SQL, Git, Linux",Computer Science degree,jessica claire montgomery street san francisco...,jessica claire montgomery street san francisco...,jessica claire montgomery street san francisco...,ResumeAtlas,1495,189,1,4,3,64,0,17
1,REAL_0002,Java Developer,"Python, SQL, Git, Linux",Computer Science degree,jared arthur maica java developer 17994568777 ...,jared arthur maica java developer linkedincomi...,jared arthur maica java developer 17994568777 ...,ResumeAtlas,1686,206,1,4,3,62,0,17
2,REAL_0003,Java Developer,"Python, SQL, Git, Linux",Computer Science degree,jessica claire 9 resumesampleexamplecom 555 43...,jessica claire 9 resumesampleexamplecom montgo...,jessica claire 9 resumesampleexamplecom 555 43...,ResumeAtlas,5555,715,1,4,3,62,0,17
3,REAL_0004,Java Developer,"Python, SQL, Git, Linux",Computer Science degree,jessica claire 9 resumesampleexamplecom 555 43...,jessica claire 9 resumesampleexamplecom montgo...,jessica claire 9 resumesampleexamplecom 555 43...,ResumeAtlas,12834,1657,1,4,3,61,0,17
4,REAL_0005,Java Developer,"Python, SQL, Git, Linux",Computer Science degree,jessica claire 100 montgomery st 10th floor xx...,jessica claire 100 montgomery st 10th floor xx...,jessica claire 100 montgomery st 10th floor xx...,ResumeAtlas,4181,489,1,4,3,57,0,17


## Step 4: Load FAISS Index and Resume Chunks

Load the vector database and stored resume chunks created in the RAG notebook.

In [6]:
index = faiss.read_index("../models/resume_index.faiss")

with open("../models/resume_chunks.pkl", "rb") as file:

    resume_chunks = pickle.load(file)

print("FAISS Index Loaded :", index.ntotal)

print("Resume Chunks :", len(resume_chunks))

FAISS Index Loaded : 2
Resume Chunks : 2


### Step 5: Load Sentence Transformer

Load the embedding model used for semantic similarity.

In [7]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding Model Loaded Successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Model Loaded Successfully


## Step 6: Select a Candidate Resume

Select one resume from the dataset for complete AI analysis.

In [8]:
resume = df.loc[25, "Clean_Text"]

print(resume[:1000])

entry level java developer reports development manager role junior java developer responsible designing developing supporting software products well work documentation specialists development team members develop features repair defects individual also support test teams customer support teams duties primary job functions produce javabased web applications support testing activities internal test teams help maintain applications write test plans perform unit test assist nightly build process minimum knowledge skills abilities bachelors degree fouryear college university related field equivalent experience must experience java sql jsp html javascript desired skills knowledge application servers jboss knowledge java ee specifically ejb knowledge relational databases oracle microsoft sql server experience jquery development library knowledge jasper report server motivated research learn new technologies tools techniques etc please submit resume elizabeth hoffmantedscom mail elizabeth hoff

### Step 7: Predict Resume Category

Predict the job category of the selected resume using the trained machine learning model.

In [9]:
from scipy.sparse import hstack

# TF-IDF Features
resume_vector = tfidf.transform([resume])

# Numerical Features
extra_features = np.array([[
    df.loc[25, "Resume_Length"],
    df.loc[25, "Word_Count"],
    df.loc[25, "Sentence_Count"],
    df.loc[25, "Skill_Count"],
    df.loc[25, "Education_Length"],
    df.loc[25, "Experience_Length"]
]])

# Combine Features
final_vector = hstack([resume_vector, extra_features])

# Prediction
prediction = classifier.predict(final_vector)

predicted_category = label_encoder.inverse_transform(prediction)[0]

print("Predicted Category :", predicted_category)

Predicted Category : Java Developer


### Step 8: Define Job Description

Create a sample job description for resume evaluation.

In [10]:
job_description = """
Python Developer

Required Skills:
- Python
- SQL
- Machine Learning
- Deep Learning
- Git
- Docker
- AWS
- REST API
- Data Analysis
- Pandas
- NumPy

Experience:
0-2 years

Education:
Bachelor's Degree in Computer Science or related field.
"""

print(job_description)


Python Developer

Required Skills:
- Python
- SQL
- Machine Learning
- Deep Learning
- Git
- Docker
- AWS
- REST API
- Data Analysis
- Pandas
- NumPy

Experience:
0-2 years

Education:
Bachelor's Degree in Computer Science or related field.



# Step 7: Extract Resume Skills

Extract skills available in the selected resume.

In [11]:
resume_skills = [
    skill.strip().lower()
    for skill in df.loc[25, "Skills"].split(",")
]

print("Resume Skills:")

resume_skills

Resume Skills:


['python', 'sql', 'git', 'linux']

# Step 8: Define Required Job Skills

Create a list of required skills from the job description.

In [12]:
jd_skills = [
    "python",
    "sql",
    "machine learning",
    "deep learning",
    "git",
    "docker",
    "aws",
    "rest api",
    "data analysis",
    "pandas",
    "numpy"
]

print("Job Description Skills:")

jd_skills

Job Description Skills:


['python',
 'sql',
 'machine learning',
 'deep learning',
 'git',
 'docker',
 'aws',
 'rest api',
 'data analysis',
 'pandas',
 'numpy']

# Step 9: Match Resume Skills with Job Description

In [13]:
matched_skills = list(
    set(resume_skills).intersection(jd_skills)
)

missing_skills = list(
    set(jd_skills) - set(resume_skills)
)

print("Matched Skills:")

print(matched_skills)

print()

print("Missing Skills:")

print(missing_skills)

Matched Skills:
['git', 'python', 'sql']

Missing Skills:
['numpy', 'aws', 'docker', 'machine learning', 'rest api', 'data analysis', 'deep learning', 'pandas']


# Step 10: Calculate ATS Score

In [14]:
ats_score = (
    len(matched_skills) / len(jd_skills)
) * 100

print(f"ATS Score : {ats_score:.2f}%")

ATS Score : 27.27%


## Step 11: Analyze Resume Strengths

Identify the strengths of the resume based on matched skills and resume statistics.

In [15]:
strengths = []

if ats_score >= 70:
    strengths.append("Good ATS Compatibility")
elif ats_score >= 40:
    strengths.append("Moderate ATS Compatibility")
else:
    strengths.append("ATS score needs improvement")

if len(matched_skills) > 0:
    strengths.append("Relevant technical skills found")

if df.loc[25, "Resume_Length"] > 1000:
    strengths.append("Resume contains sufficient information")

if df.loc[25, "Experience_Length"] > 40:
    strengths.append("Good work experience section")

print("Resume Strengths\n")

for s in strengths:
    print("✔", s)

Resume Strengths

✔ ATS score needs improvement
✔ Relevant technical skills found
✔ Resume contains sufficient information
✔ Good work experience section


## Step 12: Analyze Resume Weaknesses

Identify the missing skills and weak areas of the resume.

In [16]:
weaknesses = []

if ats_score < 70:
    weaknesses.append("Low ATS Score")

if len(missing_skills) > 0:
    weaknesses.append("Several important skills are missing")

if "machine learning" in missing_skills:
    weaknesses.append("Machine Learning knowledge is missing")

if "aws" in missing_skills:
    weaknesses.append("Cloud skills are missing")

print("Resume Weaknesses\n")

for w in weaknesses:
    print(">", w)

Resume Weaknesses

> Low ATS Score
> Several important skills are missing
> Machine Learning knowledge is missing
> Cloud skills are missing


# Step 13: Generate Personalized Feedback

Generate explainable feedback based on resume analysis.

In [17]:
feedback = f"""
Resume Analysis Report

ATS Score : {ats_score:.2f}%

Matched Skills :
{", ".join(matched_skills)}

Missing Skills :
{", ".join(missing_skills)}

Recommendation:

Improve the missing technical skills.

Update the resume with projects related to the required job profile.

Add certifications and practical experience.

Include measurable achievements wherever possible.
"""

print(feedback)


Resume Analysis Report

ATS Score : 27.27%

Matched Skills :
git, python, sql

Missing Skills :
numpy, aws, docker, machine learning, rest api, data analysis, deep learning, pandas

Recommendation:

Improve the missing technical skills.

Update the resume with projects related to the required job profile.

Add certifications and practical experience.

Include measurable achievements wherever possible.



## Step 14: Retrieve Relevant Resume Chunks

Retrieve the most relevant resume chunks using semantic search based on the job description.

In [18]:
query = job_description

query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True
)

distances, indices = index.search(
    query_embedding,
    k=2
)

print("Retrieved Chunk Indices :", indices[0])
print("Distances :", distances[0])

Retrieved Chunk Indices : [1 0]
Distances : [1.1835318 1.2751071]


## Step 15: Display Retrieved Resume Chunks

Display the retrieved resume chunks that are most relevant to the job description.

In [19]:
retrieved_context = ""

for idx in indices[0]:

    print("=" * 80)

    print(resume_chunks[idx])

    print()

    retrieved_context += resume_chunks[idx] + "\n\n"

small large projects server administrator john deere city state server administrator windows linux virtual physical servers associate technology analyst john deere parts city state developed sas programs generate report data wrote vba macros behind excel publish reports parttime student john deere city state managed access databases engine audit process published monthly reports developed internal web sites education bachelor arts management information systems 2005 university northern iowa cedar falls ia

jessica claire montgomery street san francisco ca resumesampleexamplecom professional summary highly skilled software development professional bringing 10 years software design development integration advanced knowledge java skills agile html xml jdbc tomcat work history senior java developertech lead 2014 current synnex corporation tracy ca java developer agile scrum team javascript java develop customer facing internal web applications underlying component applications wrote mainta

## Step 16: Generate Final Resume Report

Create a complete resume evaluation report using ATS score, matched skills, missing skills, strengths, weaknesses, and retrieved resume information.

In [20]:
print("=" * 70)
print("AI RESUME ANALYSIS REPORT")
print("=" * 70)

print("\nATS Score:")
print(f"{ats_score:.2f}%")

print("\nMatched Skills:")
for skill in matched_skills:
    print("✔", skill)

print("\nMissing Skills:")
for skill in missing_skills:
    print("✘", skill)

print("\nStrengths:")
for s in strengths:
    print("✔", s)

print("\nWeaknesses:")
for w in weaknesses:
    print("✘", w)

print("\nPersonalized Feedback:")
print(feedback)

print("\nRetrieved Resume Context:")
print(retrieved_context[:700])

AI RESUME ANALYSIS REPORT

ATS Score:
27.27%

Matched Skills:
✔ git
✔ python
✔ sql

Missing Skills:
✘ numpy
✘ aws
✘ docker
✘ machine learning
✘ rest api
✘ data analysis
✘ deep learning
✘ pandas

Strengths:
✔ ATS score needs improvement
✔ Relevant technical skills found
✔ Resume contains sufficient information
✔ Good work experience section

Weaknesses:
✘ Low ATS Score
✘ Several important skills are missing
✘ Machine Learning knowledge is missing
✘ Cloud skills are missing

Personalized Feedback:

Resume Analysis Report

ATS Score : 27.27%

Matched Skills :
git, python, sql

Missing Skills :
numpy, aws, docker, machine learning, rest api, data analysis, deep learning, pandas

Recommendation:

Improve the missing technical skills.

Update the resume with projects related to the required job profile.

Add certifications and practical experience.

Include measurable achievements wherever possible.


Retrieved Resume Context:
small large projects server administrator john deere city state s

## Step 17: Save Resume Analysis Report

Save the complete analysis report to a text file for future reference.

In [21]:
report = f"""
=============================
AI RESUME ANALYSIS REPORT
=============================

ATS Score:
{ats_score:.2f}%

Matched Skills:
{matched_skills}

Missing Skills:
{missing_skills}

Strengths:
{strengths}

Weaknesses:
{weaknesses}

Feedback:
{feedback}
"""

with open("../outputs/resume_report.txt", "w", encoding="utf-8") as file:
    file.write(report)

print("Resume Report Saved Successfully.")

Resume Report Saved Successfully.


## Step 18: Conclusion

The AI Resume Analyzer successfully:

- Evaluated the resume
- Calculated ATS score
- Identified matching and missing skills
- Generated strengths and weaknesses
- Retrieved relevant resume information using FAISS
- Produced an explainable resume analysis report

This completes the AI Resume Analysis Pipeline.